# Model Solution — Chi-Square from Nested Loops to Matrix Math

This notebook is the complete, runnable solution that accompanies `chi2-tutorial.md`.

Every step in the tutorial appears here as one or more cells. Run the cells in order. Use this notebook to check your work — try the tutorial yourself first.

Only two libraries are imported in this notebook: NumPy (Step 5) and `scipy.stats.chi2` (Step 12).

## Step 1 — Enter the observed data

Store the table from above as a Python list of lists. Each inner list is one row (one color). Each position inside a row is a genre. Keep the column order consistent with the table above.

In [ ]:
# Store the observed data as a Python list of lists
observed = [
    [ 9, 11, 11, 11, 11, 10],  # Red
    [ 9, 10,  9, 10,  9,  9],  # Green
    [11, 11,  9,  9,  9, 11],  # Yellow
    [10,  9, 11, 11, 11,  9],  # Purple
    [10, 10,  9, 10,  9,  9],  # Black
    [11, 10, 10,  9, 10,  9],  # Other
]

# Evaluate the observed data
observed

## Step 2 — Inspect a single cell, a row, and a column

A little hands-on indexing pays off later. Try a few lookups now.

In [ ]:
# Inspect individual cells, rows, and columns
red_books        = observed[0]      # the entire first row
purple_cookbooks = observed[3][5]   # row 3 (Purple), column 5 (Cook)

# Print, evaluate, inspect the results
print(f"All red books per genre: {red_books}")
print(f"The sum of all red books: {sum(red_books)}")
print(f"Purple cookbooks: {purple_cookbooks}")

## Step 3 — Row totals with a `for` loop

A natural first instinct is to walk through each row, sum its entries, and collect those sums. That is exactly what this function does. There is nothing wrong with this approach for a small table; it is short, readable, and correct.

In [ ]:
# Calculate row totals using a for loop
def row_totals_loop(matrix):
    """Sum each row of a matrix and return the totals as a list."""
    totals = []
    for row in matrix:
        totals.append(sum(row))
    return totals

# Print, evaluate, inspect function results
print(f"Each row total: {row_totals_loop(observed)}")
print(f"All books count: {sum(row_totals_loop(observed))}")

> **Aside — f-strings**
>
> Each `print` in Steps 2 and 3 begins with `f"..."`. That `f` prefix marks an f-string (formatted string). Within the context of an f-string, any Python expression inside the squiggly brackets `{...}` is evaluated and dropped into the string in place. Compare:
>
> - `print("Each row total:", row_totals_loop(observed))` (two arguments joined by a comma).
> - `print("Each row total: " + str(row_totals_loop(observed)))` (string concatenation using the `+` operator).
> - `print(f"Each row total: {row_totals_loop(observed)}")` — one string with the value sitting exactly where it will appear in the output.
>
> The braces accept any expression, not just a bare variable name which is why `{sum(red_books)}` and `{sum(row_totals_loop(observed))}` were valid in the code you just ran. Without the `f`, `"{x}"` is three literal characters; the prefix is what turns substitution on.
>
> Later, when learning more about f-strings, you will also see how a colon inside the braces introduces a format specification. For example, `f"{p_value:.4f}"` rounds to four decimal places, which will be useful for the p-value in Step 12. You will see f-strings again, they are the standard way to build strings everywhere in modern Python.

## Step 4 — Column totals with a nested `for` loop

Column totals are where the beginner approach starts to feel awkward. A column is not a single Python list it is the *k*-th entry of every row. To collect a column you have to loop through the rows for each column, which means a loop inside a loop.

In [ ]:
# Calculate column totals using a nested for loop
def col_totals_loop(matrix):
    """Sum each column of a matrix and return the totals as a list."""
    n_cols = len(matrix[0])
    totals = []
    for c in range(n_cols):
        running = 0
        for row in matrix:
            running += row[c]
        totals.append(running)
    return totals

# Print, evaluate, inspect function results
print(f"Column totals: {col_totals_loop(observed)}")
print(f"Col 2 toal (Science): {col_totals_loop(observed)[1]}")

## Step 5 — Bring in NumPy

So far you have only used built-in Python. Time to introduce one library: NumPy.

NumPy is the standard array library in Python data science. You will reach for it whenever you have a rectangle of numbers and want to do arithmetic on the whole thing at once. The same skills carry directly into pandas, scikit-learn, PyTorch, and JAX, so the time you invest here pays back many times over.

Two things to know about a NumPy array right now:

- It looks like a list of lists when you print it, but the whole array stores numbers in one block of memory.
- Arithmetic happens *element-wise* without any loop you write. Expressions like `arr + 1`, `arr_a - arr_b`, and `arr ** 2` operate on every cell at once.

Convert your list of lists to a NumPy array and print it.

In [ ]:
# Convert list of lists to NumPy array
import numpy as np

# Create NumPy array from list of lists
observed_arr = np.array(observed)
observed_arr

## Step 6 — Row, column, and grand totals in one line each

NumPy reads `axis=1` as "sum across the columns of each row" which is one total per each row. `axis=0` is "sum down the rows of each column" which is one total per each column. Drop the `axis` argument and you get the grand total of every cell.

In [ ]:
# Calculate row totals, column totals, and grand total using NumPy
row_t = np.sum(observed_arr, axis=1)
col_t = np.sum(observed_arr, axis=0)
grand = np.sum(observed_arr)

# Print, evaluate, inspect results using f-strings
print(f"Row totals:  {row_t}")
print(f"Col totals:  {col_t}")
print(f"Grand total: {grand}")

## Step 7 — What is the *expected* matrix?

The chi-square test starts with a question:

> If cover color and genre had no relationship at all, what would the table look like?

For any one cell of the expected matrix, the recipe is short. Multiply that cell's row total by that cell's column total, then divide by the grand total of the whole table. As code: `expected_count = (row_total * col_total) / grand_total`. The same recipe in math notation (the small `i` and `j` are just labels for which row and which column you are looking at):

$$ E_{ij} = \frac{\text{row total}_i \times \text{col total}_j}{\text{grand total}} $$

A specific example helps. The Red row crossed with the Science column has a row total of `63`, a column total of `61`, and a grand total of `356`. So the expected count for that cell is `(63 * 61) / 356`, which works out to about `10.8` books.

Why does this recipe make sense? If cover color and genre have nothing to do with each other, the count you would expect in any one cell depends only on how big its row is and how big its column is. Bigger rows and bigger columns mean bigger expected counts. Dividing by the grand total keeps the answer on the same scale as the original observed counts.

A cell where the observed count is much bigger or smaller than this expected count is a potential bit of evidence weighing against the no-relationship assumption.

## Step 8 — Expected matrix with nested `for` loops

The literal translation of the formula above is a loop inside a loop. For every row index, for every column index, compute one expected value. That is what the beginner instinct says, so write it that way first.

In [ ]:
# Calculate expected matrix using nested for loops
def expected_loop(row_t, col_t, grand):
    """Calculate expected matrix using nested for loops."""
    expected = []
    for i in range(len(row_t)):
        row = []
        for j in range(len(col_t)):
            row.append(row_t[i] * col_t[j] / grand)
        expected.append(row)
    return expected

# Print, evluate, inspect expected matrix
expected_loop_result = expected_loop(row_t, col_t, grand)
print(expected_loop_result)

## Step 9 — Expected matrix in one line of NumPy

NumPy ships a function called `np.outer` whose entire job is to take two 1-D arrays and produce the table of all pairwise products. That table is exactly the numerator of the expected-value formula.

In [ ]:
# NumPy approach using np.outer
expected = np.outer(row_t, col_t) / grand
expected

As a matter of offering a complete range of options another approach would be to use the traditional operators `*` and `/` directly on the arrays.

In [ ]:
# Alternative approach using traditional operators
expected = (row_t * col_t) / grand
expected

In [ ]:
# Verify the results match the loop-based approach
np.allclose(expected, np.array(expected_loop(row_t, col_t, grand)))

## Step 10 — The normalized error matrix

Now we compare observed to expected, cell by cell. The chi-square test does this in a specific shape:

$$ Z_{ij} = \frac{(O_{ij} - E_{ij})^2}{E_{ij}} $$

Squaring makes positive and negative deviations contribute equally (a cell that is two books low matters the same as a cell that is two books high). Dividing by the expected value puts each deviation on a relative scale. Being two off when ten were expected is a bigger deal than being two off when a thousand were expected.

The beginner version is yet another nested loop. The matrix version is a single line.

In [ ]:
# Calculate normalized error matrix
normalized_error = (observed_arr - expected) ** 2 / expected
normalized_error

## Step 11 — Sum the normalized errors

Add up every entry of the normalized-error matrix. That single number is the chi-square statistic. It is a one-number summary of how far the observed table sits from the no-relationship expectation.

In [ ]:
# Calculate chi-square statistic
chi2_stat = np.sum(normalized_error)
print(f"Chi-square statistic: {chi2_stat:.8f}")

## Step 12 — From the statistic to a p-value

A raw chi-square number is hard to interpret without a yardstick. The yardstick is the chi-square distribution, parameterized by a single integer called the degrees of freedom. For a contingency table:

$$ df = (\text{number of rows} - 1) \times (\text{number of columns} - 1) $$

The p-value is the probability of seeing a chi-square statistic at least as large as the one you got (or larger) if the null hypothesis (no relationship) was true. A small p-value is evidence against the null. SciPy has the distribution built in. This is the only second-library import in the tutorial.

In [ ]:
# Calculate p-value with scipy.stats
from scipy.stats import chi2

# Calculate degrees of freedom
n_rows, n_cols = observed_arr.shape
df = (n_rows - 1) * (n_cols - 1)

# Calculate p-value
p_value = chi2.sf(chi2_stat, df)

# Print, inspect, evaluate with f-strings
print("chi-square statistic:", chi2_stat)
print("degrees of freedom:  ", df)
print("p-value:             ", p_value)

## Step 13 — Wrap the whole pipeline as one function

Each piece works. Now compose them. This is functional style: a single function with no shared state and a single job (run the test on one observed table) built out of the named operations you already understand.

In [ ]:
# Wrap the whole pipeline as one function
def chi_square_test(observed_list):
    """
    Calculate the chi-square statistic, degrees of freedom, and p-value
    for a given observed contingency table.
    
    Parameters:
    observed_list (list): A 2D list representing the observed frequencies.
    
    Returns:
    tuple: A tuple containing the chi-square statistic, degrees of freedom, and p-value.
    """
    observed_arr = np.array(observed_list)

    row_t = np.sum(observed_arr, axis=1)    # Sum along rows
    col_t = np.sum(observed_arr, axis=0)    # Sum along columns
    grand = np.sum(observed_arr)            # Grand total

    expected         = np.outer(row_t, col_t) / grand
    normalized_error = (observed_arr - expected) ** 2 / expected
    chi2_stat        = np.sum(normalized_error)

    n_rows, n_cols = observed_arr.shape    # Get the dimensions of the array
    df = (n_rows - 1) * (n_cols - 1)       # Calculate degrees of freedom
    p_value = chi2.sf(chi2_stat, df)       # Calculate p-value

    return chi2_stat, df, p_value          # Return the results

# Print, inspect, evaluate results
chi_square_test(observed)

## Step 14 — Sanity check with a known case

Before trusting a function on real data, always run it on data whose answer you can verify by hand. Here is a deliberately lopsided 2×2 table.

In [ ]:
# Define a 2x2 table with a strong pattern
strong_pattern = [
    [30, 10],
    [10, 30],
]

# Run the chi-square test
chi_square_test(strong_pattern)

In [ ]:
# Define a 2x2 table where observed equals expected
flat = [       # No relationship between rows + columns
    [10, 10],
    [10, 10],
]

# Run the chi-square test
chi_square_test(flat)

## Where to go next

These extensions are in roughly increasing difficulty. Pick the ones that look interesting and ignore the rest.

1. Modify `chi_square_test` so it also returns the `expected` array and the `normalized_error` matrix. Then write a one-liner that prints the (row, column) of the single cell that contributes most to the statistic. This is how you find *which* cells are driving a significant result.
2. Replace `np.outer(row_t, col_t)` with broadcasting: `row_t[:, None] * col_t[None, :]`. Use `np.allclose` to confirm the two expressions agree. Broadcasting is the more general tool and shows up everywhere in NumPy, pandas, PyTorch, and JAX. Knowing it well is one of the highest-leverage skills in scientific Python.
3. Compare your answer to `scipy.stats.chi2_contingency(observed)`. You should match its first two return values to many digits. Reading that function's docstring is a good next exercise in itself.
4. Generate a 6×6 table with `np.random.randint` and run your function on it. How extreme do the counts have to be before the p-value drops below `0.05`? Try several seeds — `np.random.seed(0)`, `np.random.seed(1)`, and so on — to get a feel for how much p-values move around even when nothing is really going on.
5. Plot the normalized-error matrix as a heatmap with `matplotlib.pyplot.imshow` so cells that contribute most to the statistic light up visually. This is a fast way to read where a real relationship lives in a larger table.

A complete, runnable version of every step lives in `chi2-solution.ipynb`. Use it to check your work, not to skip ahead.